In [43]:
#导入库
import pandas as pd
import numpy as np
import sqlite3
import os
import glob
#设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option("display.width",1000)
#设置文件路径

data_path = './data/'
#读取所有数据
csv_files = glob.glob(data_path + '*.csv')
#建立字典
dataframes = {}
#遍历数据
for file in csv_files:
    name = os.path.basename(file).replace('.csv', '')  #替换将后缀替换为空
    dataframes[name] = pd.read_csv(file)

In [44]:
#认识两张表的关联关系
orders = dataframes["olist_orders_dataset"]
items = dataframes["olist_order_items_dataset"]

print("订单主表")
print(orders[["order_id","customer_id"]].head())#双【】选取多个字段返回df

print("\n 商品明细")
print(items[["order_id","order_item_id","product_id","price"]].head())

订单主表
                           order_id                       customer_id
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c

 商品明细
                           order_id  order_item_id                        product_id   price
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61   58.90
1  00018f77f2f0320c557190d7a144bdd3              1  e5f2d52b802189ee658865ca93d83a8f  239.90
2  000229ec398224ef6ca0657da4fc703e              1  c777355d18b72b67abbeef9df44fd0fd  199.00
3  00024acbcdf0a6daa1e931b038114c75              1  7634da152a4610f1595efa32f14722fc   12.99
4  00042b26cf59d7ce69dfabb4e55b4fd9              1  ac6c3623068f30de03045865e4e10089  199.90


In [45]:
#merge（）重点
merged_data = pd.merge(
    orders,#左边的订单表
    items,#右边的订单表
    on="order_id",#关于“order_id”链接
    how="inner",#内连接（保留所有匹配记录），left，左连接（保留左表匹配记录）；right，右链接（保留右表）；outer外链接，（保留两边全部记录）
)
print(merged_data.shape)
merged_data.head()
#order表8列，items表7列，关于order_id链接，14列。

(112650, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [46]:
print("订单主表行数：",len(orders))
print("商品明细表行数",len(items))
print("合并后行数",len(merged_data))

print("合并后订单数量",
      merged_data["order_id"].nunique())
#inner内连接，每个item都有对应的order，所以合并后merged_data行数等于items，订单数的偏少则是一个订单对应多个明细（order_id——》items）且nunique（）是统计不重复订单
print("重复订单数",len(merged_data)-merged_data["order_id"].nunique())

print("未匹配到商品明细订货",len(orders)-merged_data["order_id"].nunique())

订单主表行数： 99441
商品明细表行数 112650
合并后行数 112650
合并后订单数量 98666
重复订单数 13984
未匹配到商品明细订货 775


In [47]:
#检查未匹配订单状态
un_orders = orders[
    ~orders["order_id"].isin(items["order_id"])
]
print("未匹配订单",len(un_orders))
print(un_orders["order_status"].value_counts())

未匹配订单 775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [48]:
left_merged = pd.merge(
    orders,
    items,
    on="order_id",
    how="left"
)
print("左连接后行数",len(left_merged))
print("左连接后订单数量：",left_merged["order_id"].nunique())
print("没有商品明细的记录数",
      left_merged["product_id"].isna().sum())

左连接后行数 113425
左连接后订单数量： 99441
没有商品明细的记录数 775


In [49]:
#计算销售金额(商品价格+物流)【total_amount】利于后续进行计算
items["total_amount"] = (
    items["price"] + items["freight_value"]
)
print(items[[
    "order_id",
    "price",
    "freight_value",
    "total_amount"

]].head())

                           order_id   price  freight_value  total_amount
0  00010242fe8c5a6d1ba2dd792cb16214   58.90          13.29         72.19
1  00018f77f2f0320c557190d7a144bdd3  239.90          19.93        259.83
2  000229ec398224ef6ca0657da4fc703e  199.00          17.87        216.87
3  00024acbcdf0a6daa1e931b038114c75   12.99          12.79         25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9  199.90          18.14        218.04


In [50]:
#销售金额是商品价格【sales_amount】
items["sales_amount"] = items["price"]
print(items[["sales_amount","total_amount"]].head())

   sales_amount  total_amount
0         58.90         72.19
1        239.90        259.83
2        199.00        216.87
3         12.99         25.78
4        199.90        218.04


In [51]:
#按订单汇总销售金额
order_sales = items.groupby("order_id")["sales_amount"].sum()
print(order_sales.head())

order_id
00010242fe8c5a6d1ba2dd792cb16214     58.90
00018f77f2f0320c557190d7a144bdd3    239.90
000229ec398224ef6ca0657da4fc703e    199.00
00024acbcdf0a6daa1e931b038114c75     12.99
00042b26cf59d7ce69dfabb4e55b4fd9    199.90
Name: sales_amount, dtype: float64


In [52]:
print("销售总额：",order_sales.sum())#数字统计
print("订单数量:",order_sales.count())#非空统计
print("平均销售额：",order_sales.mean())

销售总额： 13591643.7
订单数量: 98666
平均销售额： 137.75407637889444


In [53]:
#price最高的10个订单
top_orders = order_sales.sort_values(
    ascending=False,
)
print(top_orders.head(10))

order_id
03caa2c082116e1d31e67e9ae3700499    13440.0
736e1922ae60d0d6a89247b851902527     7160.0
0812eb902a67711a1cb742b3cdaa65ae     6735.0
fefacc66af859508bf1a7934eab1e97f     6729.0
f5136e38d1a14a4dbd87dff67da82701     6499.0
2cc9089445046817a7539d90805e6e5a     5934.6
a96610ab360d42a2e5335a3998b4718a     4799.0
199af31afc78c699f0dbf71fb178d4d4     4690.0
b4c4b76c642808cbe472a32b86cddc95     4599.9
8dbc85d1447242f3b127dda390d56e19     4590.0
Name: sales_amount, dtype: float64


In [57]:
#统计最高销售订单
top_order_id = order_sales.idxmax()
top_order = items[
    items["order_id"] == top_order_id
]
print("最高销售额订单：",top_order_id)
print("商品件数：",len(top_order))
print(top_order[[
    "product_id",
    "price",
    "freight_value",
]])

最高销售额订单： 03caa2c082116e1d31e67e9ae3700499
商品件数： 8
                            product_id   price  freight_value
1647  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1648  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1649  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1650  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1651  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1652  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1653  5769ef0a239114ac3a854af00df129e4  1680.0          28.01
1654  5769ef0a239114ac3a854af00df129e4  1680.0          28.01


In [58]:
#在上面可以看出，这是一个商品买了8件
#现在查商品销售总额和运费总额
print("商品消费额：",top_order["price"].sum())
print("运费总额：",top_order["freight_value"].sum())
print("商品加运费",top_order["price"].sum()+
      top_order["freight_value"].sum())

商品消费额： 13440.0
运费总额： 224.08
商品加运费 13664.08
